# Média das curvas de calibração — duas rodadas

Bancada Litel / UFJF — *Painel de Bombas*. Segunda campanha em **24/09/2026**,
mesmo método da primeira (água, frascos A/B, volume lido em **B**). Cada
parâmetro (bomba × sentido) tem agora **duas medidas**. Este notebook junta as
rodadas e traça a **curva média**.

- Rodada 1: 22–23/09/2026 (`ensaios_calibracao.csv`)
- Rodada 2: 24/09/2026 (`ensaios_calibracao_rodada2.csv`)


## 1. Como se média a curva

O modelo de cada ensaio continua

$$
Q(\mathrm{PWM}) =
\begin{cases}
0 & \text{se } \mathrm{PWM} < \mathrm{PWM}_0 \\
a\,(\mathrm{PWM} - \mathrm{PWM}_0) & \text{se } \mathrm{PWM} \ge \mathrm{PWM}_0.
\end{cases}
$$

Com dois ensaios no mesmo par bomba + sentido, a curva média é a média
**ponto a ponto**:

$$
Q_\mathrm{média}(\mathrm{PWM}) = \frac{Q_1(\mathrm{PWM}) + Q_2(\mathrm{PWM})}{2}.
$$

Isso respeita zonas mortas diferentes: a média só sobe depois do menor
$\mathrm{PWM}_0$; entre os dois limiares um ensaio ainda vale zero.

Em paralelo, cada parâmetro ganha a média aritmética das duas medidas
($\overline{\mathrm{PWM}}_0$, $\bar a$, $\bar Q$, $\bar t$). O par
$(\overline{\mathrm{PWM}}_0,\ \bar a)$ é o que o painel usaria se adotasse a
média dos coeficientes. Essa reta **não coincide** com $Q_\mathrm{média}$
quando os $\mathrm{PWM}_0$ diferem — as figuras mostram as duas.


## 2. Método (igual nas duas rodadas)

Uma bomba por vez; mangueira esquerda no frasco **A**, direita no **B**;
cronômetro no deslocamento do menisco em B; volume alvo 150 ml (um ensaio da
rodada 2 registrou 149,9 ml); PWM de regime 90 %.

$\mathrm{PWM}_0$: menor PWM em que o motor gira de forma contínua. Na rodada 1
o mesmo valor foi adotado nos dois sentidos. Na rodada 2 vários pares ficaram
com $\mathrm{PWM}_0$ **próprio de cada sentido**.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.float_format", lambda v: f"{v:.4f}")
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 150,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.35,
        "font.size": 10,
    }
)

HERE = Path.cwd()
if not (HERE / "ensaios_calibracao.csv").exists():
    HERE = HERE / "scripts"

CSV1 = HERE / "ensaios_calibracao.csv"
CSV2 = HERE / "ensaios_calibracao_rodada2.csv"
FIG = HERE / "figuras"
FIG.mkdir(exist_ok=True)

r1 = pd.read_csv(CSV1, parse_dates=["data_hora"])
r2 = pd.read_csv(CSV2, parse_dates=["data_hora"])
r1["rodada"] = 1
r2["rodada"] = 2
ensaios = pd.concat([r1, r2], ignore_index=True)
ensaios


## 3. Recalcular $Q$ e $a$

$Q = 60V/t$ (ml/min) e $a = Q/(\mathrm{PWM}-\mathrm{PWM}_0)$, a partir do
cronômetro e do volume lido — não dos coeficientes arredondados do painel.


In [ ]:
def vazao_ml_min(volume_ml, tempo_s):
    return 60.0 * volume_ml / tempo_s


def coeficiente_a(q_ml_min, pwm, pwm0):
    span = pwm - pwm0
    if np.any(np.asarray(span) <= 0):
        raise ValueError("PWM de regime precisa ser maior que PWM0")
    return q_ml_min / span


def q_modelo(pwm, a, pwm0):
    pwm = np.asarray(pwm, dtype=float)
    return np.where(pwm < pwm0, 0.0, a * (pwm - pwm0))


calc = ensaios.copy()
calc["Q_ml_min"] = vazao_ml_min(calc["volume_ml"], calc["tempo_s"])
calc["a"] = coeficiente_a(calc["Q_ml_min"], calc["pwm_regime"], calc["pwm0"])

resumo = (
    calc[
        [
            "rodada",
            "bomba",
            "sentido",
            "data_hora",
            "volume_ml",
            "pwm0",
            "tempo_s",
            "Q_ml_min",
            "a",
        ]
    ]
    .sort_values(["bomba", "sentido", "rodada"])
    .reset_index(drop=True)
)
resumo


## 4. Duas medidas e a média de cada parâmetro

Uma linha por bomba × sentido: rodada 1, rodada 2 e média. O desvio relativo
é $(x_2-x_1)/x_1$.


In [ ]:
def desvio_rel(novo, antigo):
    return 100.0 * (novo - antigo) / antigo


linhas = []
for (bomba, sentido), grp in calc.groupby(["bomba", "sentido"]):
    g = grp.set_index("rodada").sort_index()
    a1, a2 = g.loc[1], g.loc[2]
    linhas.append(
        {
            "bomba": int(bomba),
            "sentido": sentido,
            "pwm0_r1": a1.pwm0,
            "pwm0_r2": a2.pwm0,
            "pwm0_media": 0.5 * (a1.pwm0 + a2.pwm0),
            "d_pwm0_%": desvio_rel(a2.pwm0, a1.pwm0),
            "t_r1_s": a1.tempo_s,
            "t_r2_s": a2.tempo_s,
            "t_media_s": 0.5 * (a1.tempo_s + a2.tempo_s),
            "Q_r1": a1.Q_ml_min,
            "Q_r2": a2.Q_ml_min,
            "Q_media": 0.5 * (a1.Q_ml_min + a2.Q_ml_min),
            "d_Q_%": desvio_rel(a2.Q_ml_min, a1.Q_ml_min),
            "a_r1": a1.a,
            "a_r2": a2.a,
            "a_media": 0.5 * (a1.a + a2.a),
            "d_a_%": desvio_rel(a2.a, a1.a),
        }
    )

media = pd.DataFrame(linhas).sort_values(["bomba", "sentido"]).reset_index(drop=True)
media


## 5. Curvas por bomba — rodadas e média

Linha clara tracejada: cada rodada. Linha cheia: $Q_\mathrm{média}$ ponto a
ponto. Pontos: $Q$ medido @ 90 %. A reta pontilhada é o modelo
$(\overline{\mathrm{PWM}}_0,\bar a)$.


In [ ]:
COR = {"direto": "#1d4ed8", "reverso": "#c2410c"}
ESTILO_RODADA = {1: (0.35, ":"), 2: (0.55, "--")}
pwm_eixo = np.linspace(0, 100, 801)

fig, eixos = plt.subplots(2, 3, figsize=(11.4, 7.0), sharex=True, sharey=True)
eixos = eixos.ravel()

for i, bomba in enumerate(range(1, 7)):
    ax = eixos[i]
    grp = calc[calc["bomba"] == bomba]
    for sentido, sub in grp.groupby("sentido"):
        cor = COR[sentido]
        qs = []
        for _, row in sub.sort_values("rodada").iterrows():
            q = q_modelo(pwm_eixo, row.a, row.pwm0)
            qs.append(q)
            alpha, ls = ESTILO_RODADA[int(row.rodada)]
            ax.plot(
                pwm_eixo,
                q,
                color=cor,
                lw=1.4,
                ls=ls,
                alpha=alpha,
                label=f"{sentido} r{int(row.rodada)}",
            )
            ax.scatter(
                [row.pwm_regime],
                [row.Q_ml_min],
                color=cor,
                s=28,
                zorder=3,
                alpha=0.85,
                edgecolors="white",
                linewidths=0.5,
            )
        q_avg = np.mean(np.vstack(qs), axis=0)
        ax.plot(pwm_eixo, q_avg, color=cor, lw=2.4, label=f"{sentido} média")
        m = media[(media["bomba"] == bomba) & (media["sentido"] == sentido)].iloc[0]
        ax.plot(
            pwm_eixo,
            q_modelo(pwm_eixo, m.a_media, m.pwm0_media),
            color=cor,
            lw=1.1,
            ls="-.",
            alpha=0.8,
        )
        ax.axvline(m.pwm0_media, color=cor, ls=":", lw=0.9, alpha=0.45)
    ax.set_title(f"P0{bomba}")
    ax.set_xlim(0, 100)
    ax.set_ylim(0, None)
    ax.legend(fontsize=6.6, loc="upper left", frameon=False, ncol=2)

for ax in eixos[3:]:
    ax.set_xlabel("PWM (%)")
for ax in (eixos[0], eixos[3]):
    ax.set_ylabel("Q (ml/min)")

fig.suptitle(
    "Curvas médias — rodada 1 (pontilhada), rodada 2 (tracejada), média (cheia)",
    y=1.01,
)
fig.tight_layout()
fig.savefig(FIG / "curvas_media_por_bomba.png", bbox_inches="tight")
plt.show()


## 6. As seis curvas médias no mesmo eixo


In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(11.4, 4.5), sharey=True)
cmap = plt.cm.tab10

for ax, sentido in zip(eixos, ("direto", "reverso")):
    for bomba in range(1, 7):
        sub = calc[(calc["bomba"] == bomba) & (calc["sentido"] == sentido)]
        qs = [q_modelo(pwm_eixo, row.a, row.pwm0) for _, row in sub.iterrows()]
        q_avg = np.mean(np.vstack(qs), axis=0)
        m = media[(media["bomba"] == bomba) & (media["sentido"] == sentido)].iloc[0]
        cor = cmap(bomba - 1)
        ax.plot(
            pwm_eixo,
            q_avg,
            color=cor,
            lw=2.2,
            label=f"P0{bomba}  PWM₀={m.pwm0_media:.1f}%",
        )
        ax.scatter([90], [m.Q_media], color=cor, s=28, zorder=3)
    ax.set_title(sentido.capitalize())
    ax.set_xlabel("PWM (%)")
    ax.set_xlim(0, 100)
    ax.set_ylim(0, None)
    ax.legend(fontsize=8, frameon=False, loc="upper left")

eixos[0].set_ylabel("Q média (ml/min)")
fig.suptitle("Comparativo das curvas médias (duas rodadas)", y=1.02)
fig.tight_layout()
fig.savefig(FIG / "comparativo_curvas_medias.png", bbox_inches="tight")
plt.show()


## 7. Repetibilidade e parâmetros médios

Barras: rodada 1, rodada 2 e média. O último painel é o desvio de $Q$ @ 90 %
entre as rodadas.


In [ ]:
fig, eixos = plt.subplots(2, 2, figsize=(11.4, 7.0))
x = np.arange(1, 7)
largura = 0.22

def barras_sentido(ax, col_r1, col_r2, col_m, titulo, ylabel):
    for j, sentido in enumerate(("direto", "reverso")):
        sub = media[media["sentido"] == sentido].set_index("bomba").loc[x]
        desloc = -0.22 if sentido == "direto" else 0.22
        ax.bar(x + desloc - largura, sub[col_r1], width=largura, color=COR[sentido], alpha=0.35, label=f"{sentido} r1")
        ax.bar(x + desloc, sub[col_r2], width=largura, color=COR[sentido], alpha=0.65, label=f"{sentido} r2")
        ax.bar(x + desloc + largura, sub[col_m], width=largura, color=COR[sentido], label=f"{sentido} média")
    ax.set_title(titulo)
    ax.set_xlabel("Bomba")
    ax.set_ylabel(ylabel)
    ax.set_xticks(x)
    ax.set_xticklabels([f"P0{i}" for i in x])
    ax.legend(fontsize=7, frameon=False, ncol=2)


barras_sentido(eixos[0, 0], "pwm0_r1", "pwm0_r2", "pwm0_media", "PWM₀", "%")
barras_sentido(eixos[0, 1], "a_r1", "a_r2", "a_media", "Coeficiente a", "ml/min / %")
barras_sentido(eixos[1, 0], "Q_r1", "Q_r2", "Q_media", "Q @ 90 %", "ml/min")

ax = eixos[1, 1]
for sentido in ("direto", "reverso"):
    sub = media[media["sentido"] == sentido].set_index("bomba").loc[x]
    desloc = -0.18 if sentido == "direto" else 0.18
    cores = [COR[sentido] if v >= 0 else "#64748b" for v in sub["d_Q_%"]]
    ax.bar(x + desloc, sub["d_Q_%"], width=0.32, color=COR[sentido], label=sentido)
ax.axhline(0, color="#64748b", lw=1)
ax.set_title("Desvio de Q entre rodadas (r2 − r1)/r1")
ax.set_xlabel("Bomba")
ax.set_ylabel("%")
ax.set_xticks(x)
ax.set_xticklabels([f"P0{i}" for i in x])
ax.legend(frameon=False, fontsize=8)

fig.tight_layout()
fig.savefig(FIG / "repetibilidade_duas_rodadas.png", bbox_inches="tight")
plt.show()

print(
    "Maior |ΔQ| entre rodadas:",
    media.loc[media["d_Q_%"].abs().idxmax(), ["bomba", "sentido", "d_Q_%"]].to_dict(),
)
print(
    "Maior |ΔPWM0| entre rodadas:",
    media.loc[media["d_pwm0_%"].abs().idxmax(), ["bomba", "sentido", "pwm0_r1", "pwm0_r2", "d_pwm0_%"]].to_dict(),
)


## 8. Pares adotáveis no painel

Média aritmética dos dois ensaios. $Q$ em 90 % da reta
$(\overline{\mathrm{PWM}}_0,\bar a)$ e $Q$ da curva média ponto a ponto
(quase iguais quando os $\mathrm{PWM}_0$ são próximos).


In [ ]:
adotado = media[["bomba", "sentido", "pwm0_media", "a_media", "Q_media"]].copy()
adotado["Q_reta_media"] = q_modelo(
    90,
    adotado["a_media"].to_numpy(),
    adotado["pwm0_media"].to_numpy(),
)
adotado["Q_curva_media"] = adotado["Q_media"]
adotado["diff_Q_modelos"] = adotado["Q_reta_media"] - adotado["Q_curva_media"]
adotado


## 9. Arquivos

Figuras em `scripts/figuras/`:

- `curvas_media_por_bomba.png`
- `comparativo_curvas_medias.png`
- `repetibilidade_duas_rodadas.png`

Dependências: `pip install -r scripts/requirements.txt`. Abra a partir de
`scripts/` para achar os CSV no diretório atual.
